# Limpeza de dados - Steam Player Data
Projeto final: Análise de Dados e BI (setor Games)

## 1. Carga dos dados
Baixar o CSV no Kaggle (jackogozaly / Steam Player Data) e enviar o arquivo quando o Colab pedir.

In [2]:
import re
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]
df_raw = pd.read_csv(nome_arquivo)
df = df_raw.copy()  # df_raw permanece intacto para comparação antes/depois

print(df.shape)
df.head()

Saving Valve_Player_Data.csv to Valve_Player_Data (1).csv
(5271, 8)


,Month_Year,Avg_players,Gain,Percent_Gain,Peak_Players,URL,Date,Game_Name
0,September 2021,512350.92,268.96,+0.05%,942519,https://steamcharts.com/app/730,2021-09-01,Counter Strike: Global Offensive
1,August 2021,512081.96,6014.60,+1.19%,802544,https://steamcharts.com/app/730,2021-08-01,Counter Strike: Global Offensive
2,July 2021,506067.36,-43279.72,-7.88%,763523,https://steamcharts.com/app/730,2021-07-01,Counter Strike: Global Offensive
3,June 2021,549347.08,-110541.81,-16.75%,929940,https://steamcharts.com/app/730,2021-06-01,Counter Strike: Global Offensive
4,May 2021,659888.89,-63457.63,-8.77%,1087197,https://steamcharts.com/app/730,2021-05-01,Counter Strike: Global Offensive


## 2. Inspeção inicial

In [3]:
df.info()
print()
print("Nulos por coluna:")
print(df.isnull().sum())
print()
print("Linhas duplicadas (linha inteira):", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5271 entries, 0 to 5270
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Month_Year    5271 non-null   object 
 1   Avg_players   5271 non-null   float64
 2   Gain          5173 non-null   float64
 3   Percent_Gain  5173 non-null   object 
 4   Peak_Players  5271 non-null   int64  
 5   URL           5271 non-null   object 
 6   Date          5271 non-null   object 
 7   Game_Name     5271 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 329.6+ KB

Nulos por coluna:
Month_Year       0
Avg_players      0
Gain            98
Percent_Gain    98
Peak_Players     0
URL              0
Date             0
Game_Name        0
dtype: int64

Linhas duplicadas (linha inteira): 0


## 3. Padronização dos nomes das colunas
Nomes em minúsculas e snake_case para evitar erros de digitação nas etapas seguintes.

In [4]:
def padronizar_coluna(nome):
    nome = nome.strip().lower()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    return nome.strip("_")

df.columns = [padronizar_coluna(c) for c in df.columns]
print(list(df.columns))

['month_year', 'avg_players', 'gain', 'percent_gain', 'peak_players', 'url', 'date', 'game_name']


In [5]:
# Ajustar as chaves conforme a saída da célula anterior, se os nomes forem diferentes
RENOMEAR = {
    "gamename": "game_name",
    "name": "game_name",
}
df = df.rename(columns=RENOMEAR)

COLUNAS_ESPERADAS = ["month_year", "avg_players", "gain", "percent_gain",
                     "peak_players", "url", "date", "game_name"]
faltando = [c for c in COLUNAS_ESPERADAS if c not in df.columns]
print("Colunas faltando:", faltando)

Colunas faltando: []


## 4. Tipos e formatação
- `date` convertida para datetime.
- `percent_gain` pode vir como texto com símbolo `%`; convertida para número.
- Nomes de jogos sem espaços extras.

In [6]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

for col in ["avg_players", "gain", "peak_players"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["percent_gain"] = (
    df["percent_gain"].astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df["percent_gain"] = pd.to_numeric(df["percent_gain"], errors="coerce")

df["game_name"] = df["game_name"].astype(str).str.strip()

df.dtypes

,0
month_year,object
avg_players,float64
gain,float64
percent_gain,float64
peak_players,int64
url,object
date,datetime64[ns]
game_name,object


## 5. Duplicados
Chave lógica: um registro por jogo por mês (`game_name` + `date`).

In [7]:
dup_chave = df.duplicated(subset=["game_name", "date"], keep=False).sum()
print("Linhas com chave (jogo, mês) repetida:", dup_chave)

antes = len(df)
df = df.drop_duplicates(subset=["game_name", "date"], keep="first")
print("Removidas:", antes - len(df))

Linhas com chave (jogo, mês) repetida: 0
Removidas: 0


## 6. Valores nulos
Os nulos de `gain` e `percent_gain` no primeiro mês de cada jogo são esperados (não há mês anterior para comparar). Nulos nas colunas principais (`avg_players`, `peak_players`, `date`, `game_name`) inviabilizam a linha e ela é removida.

In [8]:
print("Nulos antes do tratamento:")
print(df[["avg_players", "peak_players", "date", "game_name", "gain", "percent_gain"]].isnull().sum())

colunas_obrigatorias = ["avg_players", "peak_players", "date", "game_name"]
antes = len(df)
df = df.dropna(subset=colunas_obrigatorias)
print("\nLinhas removidas por nulo em coluna obrigatória:", antes - len(df))

# Recalcula gain e percent_gain a partir de avg_players, por jogo, em ordem cronológica
df = df.sort_values(["game_name", "date"]).reset_index(drop=True)
df["gain_calc"] = df.groupby("game_name")["avg_players"].diff()
df["percent_gain_calc"] = df.groupby("game_name")["avg_players"].pct_change() * 100

# Compara o valor original com o recalculado
comparavel = df.dropna(subset=["gain", "gain_calc"])
divergentes = (comparavel["gain"] - comparavel["gain_calc"]).abs() > 0.5
print("Linhas em que gain original diverge do recalculado:", divergentes.sum())

# Usa o recalculado como padrão (preenche também o primeiro mês de cada jogo com NaN esperado)
df["gain"] = df["gain_calc"]
df["percent_gain"] = df["percent_gain_calc"]
df = df.drop(columns=["gain_calc", "percent_gain_calc"])

Nulos antes do tratamento:
avg_players       0
peak_players      0
date              0
game_name         0
gain             98
percent_gain    105
dtype: int64

Linhas removidas por nulo em coluna obrigatória: 0
Linhas em que gain original diverge do recalculado: 0


## 7. Consistência lógica
O pico mensal não pode ser menor que a média mensal. Valores negativos de jogadores são inválidos.

In [9]:
inconsistentes = df[df["peak_players"] < df["avg_players"]]
print("Pico < média:", len(inconsistentes))

negativos = df[(df["avg_players"] < 0) | (df["peak_players"] < 0)]
print("Valores negativos:", len(negativos))

df = df[(df["peak_players"] >= df["avg_players"]) & (df["avg_players"] >= 0)]
print("Linhas restantes:", len(df))

Pico < média: 0
Valores negativos: 0
Linhas restantes: 5271


## 8. Outliers
Método IQR aplicado por jogo, pois a escala de jogadores varia muito entre títulos. Picos de lançamento e eventos são comportamentos reais do negócio, então os outliers são **sinalizados em coluna própria e mantidos**, não removidos.

In [10]:
def sinalizar_outliers_iqr(grupo):
    q1 = grupo.quantile(0.25)
    q3 = grupo.quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    return (grupo < limite_inf) | (grupo > limite_sup)

df["outlier_avg"] = df.groupby("game_name")["avg_players"].transform(sinalizar_outliers_iqr)

print("Outliers sinalizados:", df["outlier_avg"].sum(), "de", len(df))
print(df.groupby("game_name")["outlier_avg"].sum().sort_values(ascending=False).head(10))

Outliers sinalizados: 261 de 5271
game_name
War Thunder                      24
Brawlhalla                       16
Rocket League                    16
The Binding of Isaac: Rebirth    13
Dota 2                           11
Arma 3                           11
Garry's Mod                      10
Left 4 Dead 2                    10
Team Fortress 2                  10
Spacewar                          9
Name: outlier_avg, dtype: int64


## 9. Cobertura temporal por jogo
Jogos com poucos meses de histórico têm pouco valor para análise de tendência de queda.

In [11]:
cobertura = (
    df.groupby("game_name")
      .agg(meses=("date", "count"),
           primeiro_mes=("date", "min"),
           ultimo_mes=("date", "max"))
      .sort_values("meses")
)
print(cobertura.head(10))
print()
print("Período total:", df["date"].min().date(), "a", df["date"].max().date())
print("Jogos:", df["game_name"].nunique())

# Último mês do dataset pode estar incompleto (coleta no meio do mês)
print("\nRegistros do último mês do dataset:")
print(df[df["date"] == df["date"].max()][["game_name", "avg_players"]].head())

                                meses primeiro_mes ultimo_mes
game_name                                                    
Cookie Clicker                      1   2021-09-01 2021-09-01
Pathfinder: Wrath of Righteous      1   2021-09-01 2021-09-01
NBA 2k22                            1   2021-09-01 2021-09-01
Tales of Arise                      1   2021-09-01 2021-09-01
NARAKA: BLADEPOINT                  2   2021-08-01 2021-09-01
MIR4                                2   2021-08-01 2021-09-01
Bless Unleashed                     2   2021-08-01 2021-09-01
Forza Horizion 4                    7   2021-03-01 2021-09-01
Valheim                             8   2021-02-01 2021-09-01
Dyson Sphere Program                9   2021-01-01 2021-09-01

Período total: 2012-07-01 a 2021-09-01
Jogos: 98

Registros do último mês do dataset:
                                 game_name  avg_players
93                           7 Days to Die     18453.27
170                  ARK: Survival Evolved     39489.46
1

In [12]:
# Define o mínimo de meses para manter o jogo na análise de tendência
MIN_MESES = 12
jogos_validos = cobertura[cobertura["meses"] >= MIN_MESES].index
removidos = set(cobertura.index) - set(jogos_validos)
print("Jogos com menos de", MIN_MESES, "meses:", len(removidos))

df = df[df["game_name"].isin(jogos_validos)].reset_index(drop=True)
print("Linhas finais:", len(df))

Jogos com menos de 12 meses: 13
Linhas finais: 5205


## 10. Resumo antes e depois

In [13]:
resumo = pd.DataFrame({
    "linhas": [len(df_raw), len(df)],
    "jogos": [df_raw.iloc[:, -1].nunique(), df["game_name"].nunique()],
}, index=["bruto", "limpo"])
resumo

,linhas,jogos
bruto,5271,98
limpo,5205,85


## 11. Exportação
CSV limpo para as próximas etapas (exploração e dashboard) e dicionário de dados.

In [14]:
df = df[["game_name", "date", "month_year", "avg_players", "peak_players",
         "gain", "percent_gain", "outlier_avg", "url"]]
df.to_csv("steam_players_limpo.csv", index=False)

dicionario = pd.DataFrame({
    "coluna": ["game_name", "date", "month_year", "avg_players", "peak_players",
               "gain", "percent_gain", "outlier_avg", "url"],
    "tipo": [str(df[c].dtype) for c in df.columns],
    "descricao": [
        "Nome do jogo",
        "Primeiro dia do mês de referência",
        "Mês e ano em texto (origem SteamCharts)",
        "Média mensal de jogadores simultâneos",
        "Pico mensal de jogadores simultâneos",
        "Variação absoluta da média em relação ao mês anterior (recalculada)",
        "Variação percentual da média em relação ao mês anterior (recalculada)",
        "Sinalização de outlier por IQR dentro do jogo (mantido na base)",
        "Link da página do jogo no SteamCharts",
    ],
})
dicionario.to_excel("dicionario_dados.xlsx", index=False)

files.download("steam_players_limpo.csv")
files.download("dicionario_dados.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>